# 05 - Modelo DL Pré-treinado: DistilBERT
O DistilBERT é uma versão compacta do BERT (Bidirectional Encoder Representations from Transformers), sendo 40% menor e 60% mais rápido, mantendo 97% da performance do modelo original. Foi escolhido com base em estudos acadêmicos que demonstram sua eficácia na detecção de phishing, como Uddin et al. (2024) que obtiveram 98.48% de acurácia, e Damatie et al. (2024) que atingiram 99.29% em testes controlados (IEEE, 2024). Ao contrário dos modelos anteriores, o DistilBERT já foi pré-treinado em bilhões de textos — aqui apenas fazemos o fine-tuning para o nosso problema específico.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

print(f"PyTorch versão: {torch.__version__}")
print("Bibliotecas carregadas!")

PyTorch versão: 2.12.0+cpu
Bibliotecas carregadas!


In [3]:
# Carrega o dataset limpo
df = pd.read_csv('../data/phishing_email_limpo.csv')

# Usa uma amostra menor para o BERT não demorar demais
df_sample = df.sample(n=10000, random_state=42)

print(f"Dataset completo: {df.shape[0]} e-mails")
print(f"Amostra para treino: {df_sample.shape[0]} e-mails")
print(f"\nDistribuição da amostra:")
print(df_sample['label'].value_counts())

Dataset completo: 82403 e-mails
Amostra para treino: 10000 e-mails

Distribuição da amostra:
label
1    5135
0    4865
Name: count, dtype: int64


In [5]:
# Divide em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    df_sample['text_combined'].tolist(),
    df_sample['label'].tolist(),
    test_size=0.2,
    random_state=42
)

print(f"Treino: {len(X_train)} e-mails")
print(f"Teste:  {len(X_test)} e-mails")

Treino: 8000 e-mails
Teste:  2000 e-mails


In [7]:
# Carrega o tokenizador e modelo pré-treinado do HuggingFace
print("Baixando modelo pré-treinado... aguarde!")

tokenizer_bert = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model_bert = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2  # 2 classes: legítimo e phishing
)

print("Modelo carregado!")
print(f"Parâmetros do modelo: {sum(p.numel() for p in model_bert.parameters()):,}")

Baixando modelo pré-treinado... aguarde!


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo carregado!
Parâmetros do modelo: 66,955,010


In [8]:
# Classe para organizar os dados no formato que o PyTorch espera
class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Tokeniza cada e-mail individualmente
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Cria os datasets de treino e teste
train_dataset = EmailDataset(X_train, y_train, tokenizer_bert)
test_dataset = EmailDataset(X_test, y_test, tokenizer_bert)

# DataLoader: organiza os dados em lotes para o treinamento
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print(f"Lotes de treino: {len(train_loader)}")
print(f"Lotes de teste:  {len(test_loader)}")

Lotes de treino: 500
Lotes de teste:  125


In [9]:
# Define o dispositivo (CPU nesse caso)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")

model_bert = model_bert.to(device)

# Otimizador com taxa de aprendizado baixa (padrão para fine-tuning)
optimizer = AdamW(model_bert.parameters(), lr=2e-5)

# Treina por 3 épocas
EPOCHS = 3

for epoch in range(EPOCHS):
    model_bert.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total * 100
    print(f"Época {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_loader):.4f} - Acurácia: {acc:.2f}%")

print("\nTreinamento concluído!")

Usando: cpu
Época 1/3 - Loss: 0.1684 - Acurácia: 93.29%
Época 2/3 - Loss: 0.0447 - Acurácia: 98.49%
Época 3/3 - Loss: 0.0132 - Acurácia: 99.60%

Treinamento concluído!


In [ ]:
model_bert.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"Acurácia: {accuracy_score(all_labels, all_preds) * 100:.2f}%\n")
print("Relatório de Classificação:")
print(classification_report(all_labels, all_preds, target_names=['Legítimo', 'Phishing']))